# Notebook 01 — Limpieza y construcción de la base individual Saber 11

**Objetivo:** Cargar todos los microdatos del Saber 11 (2016-1 a 2024-2), limpiar variables,
estandarizar tipos y guardar una base individual limpia en `data/cleaned/`.

**Output:** `data/cleaned/saber11_individual_limpio.parquet` (y `.csv` opcional)

---
**Periodos cubiertos:** 2016-1, 2016-2, …, 2024-1, 2024-2 (18 archivos)

**Nota sobre columnas:** Los archivos de 2016–2021 tienen 72 columnas; los de 2022–2024 tienen 84 columnas (se agregaron variables de hábitos, alimentación y trabajo del hogar). El notebook maneja esta diferencia automáticamente.

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
import warnings
warnings.filterwarnings('ignore')

# Rutas del proyecto
RAW_DIR     = '../data/raw'
CLEANED_DIR = '../data/cleaned'
os.makedirs(CLEANED_DIR, exist_ok=True)

print('Librerías cargadas correctamente')
print(f'Pandas version: {pd.__version__}')

## 1. Exploración inicial de los archivos raw

In [ ]:
# Listar todos los archivos disponibles
archivos = sorted(glob.glob(os.path.join(RAW_DIR, 'Examen_Saber_11_*.txt')))
print(f'Archivos encontrados: {len(archivos)}')
for f in archivos:
    print(' -', os.path.basename(f))

In [ ]:
# Revisar número de columnas por archivo (para detectar diferencias estructurales)
info_archivos = []
for f in archivos:
    with open(f, encoding='latin-1') as fh:
        header = fh.readline().strip()
    cols = header.split(';')
    periodo = os.path.basename(f).replace('Examen_Saber_11_', '').replace('.txt', '')
    info_archivos.append({'archivo': os.path.basename(f), 'periodo': periodo, 'n_columnas': len(cols)})

df_info = pd.DataFrame(info_archivos)
print(df_info.to_string(index=False))

In [ ]:
# Ver columnas del primer archivo (2016-1) vs el último (2024-2)
def get_columns(filepath):
    with open(filepath, encoding='latin-1') as f:
        return f.readline().strip().split(';')

cols_2016 = set(get_columns(archivos[0]))
cols_2024 = set(get_columns(archivos[-1]))

print('--- Columnas SOLO en 2016 (eliminadas en años posteriores):')
for c in sorted(cols_2016 - cols_2024):
    print(' ', c)

print('\n--- Columnas NUEVAS en 2022+ (no presentes en 2016):')
for c in sorted(cols_2024 - cols_2016):
    print(' ', c)

print(f'\nColumnas comunes: {len(cols_2016 & cols_2024)}')

## 2. Definición de variables a conservar

Se seleccionan las variables relevantes para el análisis econométrico y espacial.
Las columnas de `COLS_SOLO_2022_PLUS` están presentes solo en los archivos más recientes
y se incluirán como `NaN` en los periodos anteriores.

In [ ]:
# -------------------------------------------------------
# Variables comunes a TODOS los periodos
# -------------------------------------------------------
COLS_COMUNES = [
    # --- Identificación ---
    'periodo',
    'estu_consecutivo',

    # --- Colegio (geografía + características) ---
    'cole_cod_depto_ubicacion',
    'cole_cod_mcpio_ubicacion',
    'cole_depto_ubicacion',
    'cole_mcpio_ubicacion',
    'cole_area_ubicacion',      # URBANO / RURAL
    'cole_calendario',          # A / B
    'cole_caracter',            # ACADÉMICO / TÉCNICO / etc.
    'cole_naturaleza',          # OFICIAL / NO OFICIAL
    'cole_jornada',             # MAÑANA / TARDE / COMPLETA / etc.
    'cole_genero',              # MIXTO / MASCULINO / FEMENINO

    # --- Estudiante ---
    'estu_genero',              # M / F
    'estu_fechanacimiento',
    'estu_grado',
    'estu_inse_individual',     # Índice NSE individual (continuo)
    'estu_nse_individual',      # NSE individual (categoría)
    'estu_nse_establecimiento', # NSE del establecimiento
    'estu_discapacidad',
    'estu_etnia',
    'estu_tieneetnia',
    'estu_nacionalidad',

    # --- Familia ---
    'fami_estratovivienda',     # Estrato 1–6
    'fami_educacionmadre',
    'fami_educacionpadre',
    'fami_cuartoshogar',
    'fami_personashogar',
    'fami_numlibros',
    'fami_tieneautomovil',
    'fami_tienecomputador',
    'fami_tieneinternet',
    'fami_tienelavadora',
    'fami_tieneserviciotv',

    # --- Puntajes ---
    'punt_matematicas',
    'punt_lectura_critica',
    'punt_c_naturales',
    'punt_ingles',
    'punt_sociales_ciudadanas',
    'punt_global',

    # --- Percentiles ---
    'percentil_global',
    'percentil_matematicas',
    'percentil_lectura_critica',

    # --- Desempeño (nivel cualitativo) ---
    'desemp_matematicas',
    'desemp_lectura_critica',
]

# -------------------------------------------------------
# Variables presentes SOLO en 2022+
# Se incluirán como NaN en años anteriores
# -------------------------------------------------------
COLS_SOLO_2022_PLUS = [
    'estu_horassemanatrabaja',
    'fami_situacioneconomica',
    'fami_tienemotocicleta',
    'fami_trabajolabormadre',
    'fami_trabajolaborpadre',
]

TODAS_LAS_COLS = COLS_COMUNES + COLS_SOLO_2022_PLUS
print(f'Total de variables a conservar: {len(TODAS_LAS_COLS)}')

## 3. Carga y unión de todos los archivos

In [ ]:
def cargar_archivo(filepath, cols_deseadas):
    """
    Carga un archivo del Saber 11, selecciona las columnas disponibles
    y rellena con NaN las columnas que no existen en ese periodo.
    """
    df = pd.read_csv(
        filepath,
        sep=';',
        encoding='latin-1',
        low_memory=False,
        dtype=str  # Cargar todo como texto primero; convertir tipos después
    )
    # Normalizar nombres de columnas
    df.columns = df.columns.str.strip().str.lower()

    # Conservar solo columnas deseadas que existan; agregar NaN para las que falten
    cols_presentes  = [c for c in cols_deseadas if c in df.columns]
    cols_faltantes  = [c for c in cols_deseadas if c not in df.columns]

    df = df[cols_presentes].copy()
    for c in cols_faltantes:
        df[c] = np.nan

    return df[cols_deseadas]  # Ordenar igual en todos los periodos


# Cargar todos los archivos
dfs = []
for f in archivos:
    nombre = os.path.basename(f)
    df_tmp = cargar_archivo(f, TODAS_LAS_COLS)
    dfs.append(df_tmp)
    print(f'{nombre}: {len(df_tmp):,} filas')

# Unir en un único DataFrame
df = pd.concat(dfs, ignore_index=True)
print(f'\n==> Base combinada: {len(df):,} filas x {df.shape[1]} columnas')

## 4. Limpieza de tipos y valores

In [ ]:
# ----------------------------------------------------------
# 4.1  Columnas numéricas: puntajes, percentiles, NSE, INSE
# ----------------------------------------------------------
COLS_NUMERICAS = [
    'punt_matematicas', 'punt_lectura_critica', 'punt_c_naturales',
    'punt_ingles', 'punt_sociales_ciudadanas', 'punt_global',
    'percentil_global', 'percentil_matematicas', 'percentil_lectura_critica',
    'estu_inse_individual',
    'cole_cod_depto_ubicacion', 'cole_cod_mcpio_ubicacion',
]

for col in COLS_NUMERICAS:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print('Tipos numéricos asignados')

In [ ]:
# ----------------------------------------------------------
# 4.2  Periodo: separar año y semestre
# ----------------------------------------------------------
df['periodo'] = df['periodo'].astype(str).str.strip()
df['anio']     = df['periodo'].str[:4].astype(int)
df['semestre'] = df['periodo'].str[4].astype(int)

print('Años presentes:', sorted(df['anio'].unique()))
print('Semestres:',      sorted(df['semestre'].unique()))

In [ ]:
# ----------------------------------------------------------
# 4.3  Fecha de nacimiento → edad aproximada
# ----------------------------------------------------------
df['estu_fechanacimiento'] = pd.to_datetime(
    df['estu_fechanacimiento'], format='%d/%m/%Y', errors='coerce'
)
# Año de referencia: año del examen
df['edad_aprox'] = df['anio'] - df['estu_fechanacimiento'].dt.year

# Filtrar edades implausibles (menores de 13 o mayores de 25)
mask_edad_invalida = (df['edad_aprox'] < 13) | (df['edad_aprox'] > 25)
print(f'Registros con edad implausible: {mask_edad_invalida.sum():,}')
df.loc[mask_edad_invalida, 'edad_aprox'] = np.nan

In [ ]:
# ----------------------------------------------------------
# 4.4  Estrato: extraer número del texto (ej. 'Estrato 2' → 2)
# ----------------------------------------------------------
df['estrato'] = (
    df['fami_estratovivienda']
    .astype(str)
    .str.extract(r'(\d)')
    [0]
    .astype(float)
)
print('Distribución de estrato:')
print(df['estrato'].value_counts(dropna=False).sort_index())

In [ ]:
# ----------------------------------------------------------
# 4.5  Variables binarias Si/No → 1/0
# ----------------------------------------------------------
COLS_SINO = [
    'fami_tieneautomovil', 'fami_tienecomputador', 'fami_tieneinternet',
    'fami_tienelavadora', 'fami_tieneserviciotv', 'fami_tienemotocicleta',
]

mapa_sino = {'Si': 1, 'SI': 1, 'si': 1, 'S': 1,
             'No': 0, 'NO': 0, 'no': 0, 'N': 0}

for col in COLS_SINO:
    df[col] = df[col].map(mapa_sino).astype('Int8')  # Int8 soporta NaN

print('Variables binarias convertidas')

In [ ]:
# ----------------------------------------------------------
# 4.6  Género del estudiante: estandarizar M/F
# ----------------------------------------------------------
df['estu_genero'] = df['estu_genero'].str.strip().str.upper()
print('Valores únicos de género:', df['estu_genero'].value_counts(dropna=False).to_dict())

In [ ]:
# ----------------------------------------------------------
# 4.7  Normalizar columnas de texto: strip + upper
# ----------------------------------------------------------
COLS_TEXTO = [
    'cole_area_ubicacion', 'cole_calendario', 'cole_caracter',
    'cole_naturaleza', 'cole_jornada', 'cole_genero',
    'cole_depto_ubicacion', 'cole_mcpio_ubicacion',
    'estu_nacionalidad', 'estu_discapacidad', 'estu_tieneetnia',
    'fami_educacionmadre', 'fami_educacionpadre',
    'fami_cuartoshogar', 'fami_personashogar', 'fami_numlibros',
    'estu_nse_individual', 'estu_nse_establecimiento',
    'desemp_matematicas', 'desemp_lectura_critica',
]

for col in COLS_TEXTO:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.upper()
        .replace({'NAN': np.nan, 'NONE': np.nan, '': np.nan})
    )

print('Columnas de texto normalizadas')

In [ ]:
# ----------------------------------------------------------
# 4.8  Filtrar solo registros válidos:
#      - solo grado 11 (los microdatos también incluyen grado 9 en algunos periodos)
#      - puntaje global no nulo
# ----------------------------------------------------------
n_antes = len(df)

df['estu_grado'] = df['estu_grado'].astype(str).str.strip()
df = df[df['estu_grado'] == '11'].copy()
df = df[df['punt_global'].notna()].copy()

print(f'Filas antes del filtro: {n_antes:,}')
print(f'Filas después del filtro (grado 11 + puntaje válido): {len(df):,}')
print(f'Registros eliminados: {n_antes - len(df):,}')

## 5. Diagnóstico de calidad de la base limpia

In [ ]:
# Filas por periodo
print('Registros por periodo:')
print(df.groupby(['anio', 'semestre']).size().reset_index(name='n_estudiantes').to_string(index=False))

In [ ]:
# Porcentaje de valores faltantes por columna
miss = (df.isna().sum() / len(df) * 100).round(2).sort_values(ascending=False)
miss_df = miss[miss > 0].reset_index()
miss_df.columns = ['variable', 'pct_faltante']
print('Variables con valores faltantes:')
print(miss_df.to_string(index=False))

In [ ]:
# Estadísticas descriptivas de puntajes
PUNTAJES = ['punt_global', 'punt_matematicas', 'punt_lectura_critica',
            'punt_c_naturales', 'punt_ingles', 'punt_sociales_ciudadanas']

df[PUNTAJES].describe().round(2)

In [ ]:
# Distribución de estrato
print('Distribución de estrato (%):')
print((df['estrato'].value_counts(dropna=False, normalize=True) * 100).round(2).sort_index())

In [ ]:
# Distribución por naturaleza del colegio
print('Distribución por naturaleza del colegio (%):')
print((df['cole_naturaleza'].value_counts(dropna=False, normalize=True) * 100).round(2))

print('\nDistribución por área del colegio (%):')
print((df['cole_area_ubicacion'].value_counts(dropna=False, normalize=True) * 100).round(2))

In [ ]:
# Vista previa de las primeras filas
print('Forma final de la base:', df.shape)
df.head(3)

In [ ]:
# Tipos de datos finales
print(df.dtypes)

## 6. Guardado de la base limpia

In [ ]:
# Guardar en formato Parquet (eficiente para análisis posteriores)
ruta_parquet = os.path.join(CLEANED_DIR, 'saber11_individual_limpio.parquet')
df.to_parquet(ruta_parquet, index=False)
print(f'Base guardada en: {ruta_parquet}')
print(f'Tamaño del archivo: {os.path.getsize(ruta_parquet) / 1e6:.1f} MB')

# Opcional: guardar también en CSV (descomenta si necesitas)
# ruta_csv = os.path.join(CLEANED_DIR, 'saber11_individual_limpio.csv')
# df.to_csv(ruta_csv, index=False, encoding='utf-8-sig')
# print(f'CSV guardado en: {ruta_csv}')

In [ ]:
# Verificar lectura del archivo guardado
df_check = pd.read_parquet(ruta_parquet)
print(f'Verificación: {df_check.shape[0]:,} filas x {df_check.shape[1]} columnas')
print('Columnas disponibles:')
print(list(df_check.columns))

## 7. Resumen ejecutivo

| Item | Valor |
|------|-------|
| Periodos | 2016-1 a 2024-2 (18 archivos) |
| Archivo de salida | `data/cleaned/saber11_individual_limpio.parquet` |
| Nivel de análisis | Individual (un registro por estudiante-periodo) |
| Variables clave | Puntajes, estrato, educación padres, área/naturaleza colegio, municipio/depto |

**Próximo paso:** `02_construccion_base.ipynb` — Agregar a nivel municipal para unir con datos espaciales.